# 4.4 Spanning Sets and Linear Independence

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4 — Vector Spaces**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply
from linalg_core.elimination import solve, to_rref
from linalg_core.vecspace import is_independent, is_in_span
from linalg_core import EPSILON

import numpy as np


def vec_approx_zero(v):
    """Check if a vector (list of floats) is approximately the zero vector."""
    return all(abs(x) < EPSILON for x in v)


def vec_str(v):
    """Pretty-print a vector."""
    return "[" + ", ".join(f"{x:.4f}" for x in v) + "]"


def linear_combo(coeffs, vectors):
    """Compute c1*v1 + c2*v2 + ... + ck*vk."""
    n = len(vectors[0])
    result = [0.0] * n
    for c, v in zip(coeffs, vectors):
        for i in range(n):
            result[i] += c * v[i]
    return result


print("Setup complete.")

---

## 2. Motivation — The Redundant Sensor Problem

Imagine we have **4 sensors** in $\mathbb{R}^3$ that produce measurement vectors:

$$
\mathbf{s}_1 = \begin{bmatrix} 1 \\ 0 \\ 2 \end{bmatrix}, \quad
\mathbf{s}_2 = \begin{bmatrix} 0 \\ 1 \\ -1 \end{bmatrix}, \quad
\mathbf{s}_3 = \begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix}, \quad
\mathbf{s}_4 = \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix}.
$$

**Question:** Are any of these sensors redundant? Can we remove one without losing
information?

The **pigeonhole principle** gives us an immediate hint: we have 4 vectors in
$\mathbb{R}^3$, and $\dim(\mathbb{R}^3) = 3$. So at most 3 of them can be
linearly independent — at least one must be expressible as a combination of
the others.

But *which* one(s)? And how do we find the dependency? This is what linear
independence and spanning sets are designed to answer.

---

## 3. Build

We develop the theory of linear independence and spanning sets from first
principles, connecting each concept back to solving homogeneous systems.

### 3.1 Linear Independence — Definition

A set of vectors $\{\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k\}$ in a vector space $V$
is **linearly independent** if the only solution to

$$
c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_k \mathbf{v}_k = \mathbf{0}
$$

is the **trivial solution** $c_1 = c_2 = \cdots = c_k = 0$.

If there exists a **nontrivial** solution (at least one $c_i \neq 0$), the set is
**linearly dependent**.

**First-principles reasoning:**
- Independence means no vector in the set can be written as a linear combination
  of the others.
- Dependence means at least one vector is "redundant" — it carries no new directional
  information beyond what the other vectors already provide.
- If $c_j \neq 0$, we can isolate $\mathbf{v}_j = -\frac{1}{c_j}\sum_{i \neq j} c_i \mathbf{v}_i$,
  showing $\mathbf{v}_j$ is a combination of the rest.

**Example.** Consider $\mathbf{v}_1 = [1, 0]$ and $\mathbf{v}_2 = [0, 1]$ in $\mathbb{R}^2$.

$$
c_1 \begin{bmatrix} 1 \\ 0 \end{bmatrix} + c_2 \begin{bmatrix} 0 \\ 1 \end{bmatrix} =
\begin{bmatrix} 0 \\ 0 \end{bmatrix} \implies c_1 = 0, \; c_2 = 0.
$$

Only the trivial solution exists, so the standard basis vectors are independent.

In [ ]:
v1 = [1, 0]
v2 = [0, 1]

indep, dep = is_independent([v1, v2])
print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"\nIndependent? {indep}")
print(f"Dependency:  {dep}")
print("\nAs expected: the standard basis of R^2 is linearly independent.")

**Dependent example.** Now consider $\mathbf{v}_1 = [1, 2]$ and $\mathbf{v}_2 = [2, 4]$.
Notice $\mathbf{v}_2 = 2\mathbf{v}_1$, so $(-2)\mathbf{v}_1 + (1)\mathbf{v}_2 = \mathbf{0}$
is a nontrivial solution.

In [ ]:
v1 = [1, 2]
v2 = [2, 4]

indep, dep = is_independent([v1, v2])
print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"\nIndependent? {indep}")
print(f"Dependency:  {dep}")

combo = linear_combo(dep, [v1, v2])
print(f"\nVerify: {dep[0]:.1f}*v1 + {dep[1]:.1f}*v2 = {vec_str(combo)}")
print(f"Is zero? {vec_approx_zero(combo)}")

### 3.2 Testing via the Homogeneous System

The independence question reduces to a concrete computation: **solving a
homogeneous system**.

Stack the vectors as **columns** of a matrix $A$:

$$
A = \begin{bmatrix} \mathbf{v}_1 & \mathbf{v}_2 & \cdots & \mathbf{v}_k \end{bmatrix}
$$

Then $c_1 \mathbf{v}_1 + \cdots + c_k \mathbf{v}_k = \mathbf{0}$ is the same as
$A\mathbf{c} = \mathbf{0}$, where $\mathbf{c} = [c_1, \ldots, c_k]^T$.

**Algorithm:**
1. Form $A$ by stacking vectors as columns.
2. Reduce $A$ to RREF.
3. If every column is a pivot column $\Rightarrow$ only trivial solution $\Rightarrow$ **independent**.
4. If some column is free $\Rightarrow$ nontrivial solutions exist $\Rightarrow$ **dependent**.

Let's see this step by step with a 3-vector example in $\mathbb{R}^3$.

In [ ]:
v1 = [1, 2, 3]
v2 = [4, 5, 6]
v3 = [7, 8, 9]

A = Matrix.from_lists([
    [v1[0], v2[0], v3[0]],
    [v1[1], v2[1], v3[1]],
    [v1[2], v2[2], v3[2]],
])

print("A (vectors as columns):")
print(A)

R = A.copy()
pivots = to_rref(R)
print("\nRREF:")
print(R)
print(f"\nPivot positions: {pivots}")
print(f"Number of pivots: {len(pivots)}")
print(f"Number of vectors: 3")

if len(pivots) < 3:
    free_cols = [j for j in range(3) if j not in {c for _, c in pivots}]
    print(f"Free columns: {free_cols} -> DEPENDENT")
else:
    print("All columns are pivot columns -> INDEPENDENT")

print("\nNote: [1,2,3], [4,5,6], [7,8,9] lie on a plane.")
print("v3 = 2*v2 - v1 = [2*4-1, 2*5-2, 2*6-3] = [7, 8, 9]. Dependent!")

### 3.3 Using `is_independent` — The Library Function

Our `linalg_core.vecspace.is_independent(vectors)` function automates the
procedure above. It returns a tuple `(bool, dependency_or_None)` where:

- `True, None` means the vectors are linearly independent.
- `False, [c1, c2, ..., ck]` means the vectors are dependent, and the list
  gives coefficients such that $c_1\mathbf{v}_1 + \cdots + c_k\mathbf{v}_k = \mathbf{0}$.

Let's test with several sets of vectors in $\mathbb{R}^3$.

In [ ]:
test_sets = [
    ("Standard basis e1, e2, e3",
     [[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
    ("Two independent vectors",
     [[1, 0, 0], [0, 1, 0]]),
    ("Three collinear vectors",
     [[1, 2, 3], [2, 4, 6], [3, 6, 9]]),
    ("Three coplanar vectors",
     [[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    ("Three genuinely independent vectors",
     [[1, 0, 0], [1, 1, 0], [1, 1, 1]]),
]

for name, vecs in test_sets:
    indep, dep = is_independent(vecs)
    print(f"{name}:")
    for i, v in enumerate(vecs):
        print(f"  v{i+1} = {v}")
    print(f"  Independent? {indep}")
    if dep is not None:
        print(f"  Dependency:  {vec_str(dep)}")
        combo = linear_combo(dep, vecs)
        print(f"  Verify combo = 0: {vec_approx_zero(combo)}")
    print()

### 3.4 The Sensor Problem — Solved

Now let's return to our motivating problem. We have 4 sensor vectors in $\mathbb{R}^3$:

$$
\mathbf{s}_1 = [1, 0, 2], \quad
\mathbf{s}_2 = [0, 1, -1], \quad
\mathbf{s}_3 = [2, 1, 3], \quad
\mathbf{s}_4 = [1, 1, 1].
$$

**Step 1:** Test independence of all four.

In [ ]:
s1 = [1, 0, 2]
s2 = [0, 1, -1]
s3 = [2, 1, 3]
s4 = [1, 1, 1]

sensors = [s1, s2, s3, s4]

indep, dep = is_independent(sensors)
print("All four sensors:")
for i, s in enumerate(sensors):
    print(f"  s{i+1} = {s}")
print(f"\nIndependent? {indep}")
print(f"Dependency coefficients: {vec_str(dep)}")

combo = linear_combo(dep, sensors)
print(f"\nVerify: {dep[0]:.4f}*s1 + {dep[1]:.4f}*s2 + {dep[2]:.4f}*s3 + {dep[3]:.4f}*s4")
print(f"       = {vec_str(combo)}")
print(f"Is zero? {vec_approx_zero(combo)}")

**Step 2:** Find a maximal independent subset.

The dependency tells us these 4 vectors in $\mathbb{R}^3$ only span a 2-dimensional
subspace (since $\mathbf{s}_3 = 2\mathbf{s}_1 + \mathbf{s}_2$ and
$\mathbf{s}_4 = \mathbf{s}_1 + \mathbf{s}_2$). So removing just one vector may still
leave a dependent set. We need a **greedy algorithm**: process vectors in order,
keeping each one only if it is independent of those already kept.

In [ ]:
print("Greedy maximal independent subset:")
print("Process sensors in order, keep if independent of those already kept.\n")

kept = []
kept_labels = []

for i, s in enumerate(sensors):
    candidate = kept + [s]
    indep_check, _ = is_independent(candidate)
    status = "KEEP" if indep_check else "SKIP (redundant)"
    print(f"  s{i+1} = {s} -> {status}")
    if indep_check:
        kept.append(s)
        kept_labels.append(f"s{i+1}")

print(f"\nMaximal independent subset: {{{', '.join(kept_labels)}}}")
for label, v in zip(kept_labels, kept):
    print(f"  {label} = {v}")

indep_final, _ = is_independent(kept)
print(f"\nSubset independent? {indep_final}")
print(f"Subset size: {len(kept)} (= rank of the original set)")

print("\n--- Express each removed sensor as a combination of the kept ones ---")
for i, s in enumerate(sensors):
    if f"s{i+1}" not in kept_labels:
        in_span, coeffs = is_in_span(s, kept)
        print(f"\ns{i+1} = {s} is in span of {{{', '.join(kept_labels)}}}? {in_span}")
        if coeffs:
            recon = linear_combo(coeffs, kept)
            terms = " + ".join(f"{c:.4f}*{lbl}" for c, lbl in zip(coeffs, kept_labels))
            print(f"  {terms} = {vec_str(recon)}")

### 3.5 Key Fact — Dimension Bound

**Theorem (Larson, Sec. 4.4):** If $V$ is a vector space of dimension $n$, then:

1. Any set of **more than** $n$ vectors in $V$ is linearly **dependent**.
2. Any linearly independent set of **exactly** $n$ vectors is a **basis** for $V$.
3. Any spanning set with **exactly** $n$ vectors is a **basis** for $V$.

**First-principles reasoning:** $\mathbb{R}^n$ has dimension $n$. If you have $k > n$
vectors, stacking them as columns gives an $n \times k$ matrix. Since $k > n$,
the RREF has at most $n$ pivot columns but $k$ columns total, so at least
$k - n \geq 1$ free variables. Free variables $\Rightarrow$ nontrivial solution to
$A\mathbf{c} = \mathbf{0}$ $\Rightarrow$ the set is dependent.

This is exactly what happened with our 4 sensors in $\mathbb{R}^3$: $4 > 3$, so
dependence was guaranteed before we even computed anything.

In [ ]:
print("Dimension bound demonstration")
print("=" * 50)

for n in range(2, 5):
    for k in [n - 1, n, n + 1, n + 2]:
        if k < 1:
            continue
        np.random.seed(42 + n * 10 + k)
        vecs = [list(np.random.randn(n)) for _ in range(k)]
        indep, _ = is_independent(vecs)
        status = "INDEPENDENT" if indep else "DEPENDENT"
        bound = "(k <= n)" if k <= n else "(k > n, MUST be dependent)"
        print(f"  R^{n}: {k} random vectors -> {status:11s}  {bound}")
    print()

### 3.6 Spanning Sets

A set $S = \{\mathbf{v}_1, \ldots, \mathbf{v}_k\}$ **spans** a vector space $V$ if
every vector in $V$ can be written as a linear combination of vectors in $S$:

$$
\forall\; \mathbf{w} \in V, \quad \exists\; c_1, \ldots, c_k \quad \text{such that} \quad
\mathbf{w} = c_1 \mathbf{v}_1 + \cdots + c_k \mathbf{v}_k.
$$

For $\mathbb{R}^n$, a practical test is:
- Stack the vectors as columns of a matrix $A$.
- $S$ spans $\mathbb{R}^n$ if and only if $\text{rank}(A) = n$.
- Equivalently, the RREF of $A$ must have a pivot in every **row**.

Let's implement this.

In [ ]:
def is_spanning_set(vectors, n):
    """Test if vectors span R^n.

    Stacks vectors as columns, reduces to RREF, checks that there is a pivot
    in every row (i.e., rank = n).

    Returns (spans: bool, rank: int).
    """
    if not vectors:
        return n == 0, 0

    k = len(vectors)
    A = Matrix(n, k)
    for j, v in enumerate(vectors):
        for i in range(n):
            A[i, j] = v[i]

    R = A.copy()
    pivots = to_rref(R)
    r = len(pivots)

    return r == n, r


print("--- Spanning set tests in R^3 ---\n")

sets_to_test = [
    ("Standard basis {e1, e2, e3}",
     [[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
    ("Two vectors (cannot span R^3)",
     [[1, 0, 0], [0, 1, 0]]),
    ("Three coplanar vectors",
     [[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    ("Four vectors (overcomplete, but spanning)",
     [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 1]]),
    ("Three independent non-standard vectors",
     [[1, 0, 0], [1, 1, 0], [1, 1, 1]]),
]

for name, vecs in sets_to_test:
    spans, r = is_spanning_set(vecs, 3)
    print(f"{name}:")
    for v in vecs:
        print(f"  {v}")
    print(f"  rank = {r}, spans R^3? {spans}")
    print()

We can also test whether specific vectors are in the span using `is_in_span`.
This is useful to check reachability: can a particular target be expressed
using our spanning set?

In [ ]:
basis = [[1, 0, 2], [0, 1, -1]]

targets = [
    [2, 1, 3],
    [1, 1, 1],
    [3, -1, 7],
    [0, 0, 1],
]

print("Spanning vectors:")
for v in basis:
    print(f"  {v}")
print()

for t in targets:
    in_span, coeffs = is_in_span(t, basis)
    print(f"w = {t}")
    print(f"  In span? {in_span}")
    if coeffs:
        recon = linear_combo(coeffs, basis)
        print(f"  Coefficients: {vec_str(coeffs)}")
        print(f"  Reconstruction: {vec_str(recon)}")
    print()

### 3.7 Wronskian — Independence in Function Spaces

The concepts of independence and span extend beyond $\mathbb{R}^n$ to **function
spaces**. For example, is the set $\{1, x, x^2\}$ linearly independent in the
space of polynomials?

For functions $f_1, f_2, \ldots, f_k$ that are $(k-1)$-times differentiable, the
**Wronskian** provides a test:

$$
W(f_1, \ldots, f_k)(x) = \det \begin{bmatrix}
f_1(x)   & f_2(x)   & \cdots & f_k(x)   \\
f_1'(x)  & f_2'(x)  & \cdots & f_k'(x)  \\
\vdots   & \vdots   & \ddots & \vdots   \\
f_1^{(k-1)}(x) & f_2^{(k-1)}(x) & \cdots & f_k^{(k-1)}(x)
\end{bmatrix}
$$

If $W \neq 0$ for some $x$, the functions are linearly independent.

**Example:** Test $\{e^x, e^{2x}, e^{3x}\}$ using the Wronskian.

In [ ]:
import math
from linalg_core.determinant import det_cofactor


def wronskian_exp(coeffs, x):
    """Compute the Wronskian of {e^(a1*x), e^(a2*x), ..., e^(ak*x)} at point x.

    For f(x) = e^(a*x), the n-th derivative is a^n * e^(a*x).
    """
    k = len(coeffs)
    W = Matrix(k, k)
    for j, a in enumerate(coeffs):
        for i in range(k):
            W[i, j] = (a ** i) * math.exp(a * x)
    return det_cofactor(W), W


coeffs = [1, 2, 3]
x_val = 0.0

det_W, W = wronskian_exp(coeffs, x_val)
print(f"Functions: e^x, e^(2x), e^(3x)")
print(f"Evaluate Wronskian at x = {x_val}:\n")
print("W =")
print(W)
print(f"\ndet(W) = {det_W:.6f}")
print(f"\ndet(W) != 0? {abs(det_W) > EPSILON}")
print("Therefore {e^x, e^(2x), e^(3x)} are linearly independent.")

print("\n--- Dependent example: {e^x, 2*e^x, 3*e^x} ---")
print("These are scalar multiples of the same function,")
print("so they must be dependent. Checking via evaluation at 3 points:\n")

vecs_at_points = []
for x_val in [0.0, 1.0, 2.0]:
    row = [math.exp(x_val), 2 * math.exp(x_val), 3 * math.exp(x_val)]
    vecs_at_points.append(row)

eval_matrix = Matrix.from_lists(vecs_at_points)
print("Evaluation matrix (rows = points, cols = functions):")
print(eval_matrix)
print(f"\ndet = {det_cofactor(eval_matrix):.6f}")
print("Determinant is 0 -> dependent (as expected).")

---

## 4. Verify

We run systematic checks to confirm our implementations are correct,
and cross-validate against NumPy's `np.linalg.matrix_rank`.

In [ ]:
print("=" * 60)
print("VERIFICATION 1: Independent sets -> only trivial solution")
print("=" * 60)

np.random.seed(42)
for trial in range(5):
    n = 4
    vecs = [list(np.random.randn(n)) for _ in range(n)]
    indep, dep = is_independent(vecs)
    print(f"  Trial {trial + 1}: {n} random vectors in R^{n} -> Independent? {indep}  (dep = {dep})")

print()

In [ ]:
print("=" * 60)
print("VERIFICATION 2: Dependent sets -> nontrivial solution, combo = 0")
print("=" * 60)

np.random.seed(99)
for trial in range(5):
    n = 3
    v1 = list(np.random.randn(n))
    v2 = list(np.random.randn(n))
    v3 = [2 * a + 3 * b for a, b in zip(v1, v2)]
    vecs = [v1, v2, v3]

    indep, dep = is_independent(vecs)
    combo = linear_combo(dep, vecs) if dep else None
    is_zero = vec_approx_zero(combo) if combo else False
    print(f"  Trial {trial + 1}: v3 = 2*v1 + 3*v2 -> Independent? {indep}, combo=0? {is_zero}")

print()

In [ ]:
print("=" * 60)
print("VERIFICATION 3: Cross-check rank against np.linalg.matrix_rank")
print("=" * 60)

np.random.seed(77)
all_match = True

for trial in range(10):
    n = np.random.randint(2, 6)
    k = np.random.randint(1, 7)
    vecs = [list(np.random.randn(n)) for _ in range(k)]

    A_np = np.array(vecs).T
    np_rank = np.linalg.matrix_rank(A_np)

    _, our_rank = is_spanning_set(vecs, n)

    match = our_rank == np_rank
    if not match:
        all_match = False
    print(f"  Trial {trial + 1}: {k} vecs in R^{n} -> our rank={our_rank}, numpy rank={np_rank}  {'OK' if match else 'MISMATCH'}")

print(f"\nAll ranks match numpy? {all_match}")
print()

In [ ]:
print("=" * 60)
print("VERIFICATION 4: Spanning set + independence = basis")
print("=" * 60)

np.random.seed(123)
for trial in range(5):
    n = 3
    vecs = [list(np.random.randn(n)) for _ in range(n)]

    indep, _ = is_independent(vecs)
    spans, r = is_spanning_set(vecs, n)
    is_basis = indep and spans
    print(f"  Trial {trial + 1}: indep={indep}, spans={spans}, basis={is_basis} (rank={r})")

print("\nExpected: n random vectors in R^n are almost surely a basis.")
print()

In [ ]:
print("=" * 60)
print("VERIFICATION 5: is_in_span consistency with is_spanning_set")
print("=" * 60)

np.random.seed(55)
spanning_vecs = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
non_spanning = [[1, 0, 0], [0, 1, 0]]

for trial in range(5):
    w = list(np.random.randn(3))

    in_full, c_full = is_in_span(w, spanning_vecs)
    in_partial, c_partial = is_in_span(w, non_spanning)

    print(f"  Trial {trial + 1}: w = {vec_str(w)}")
    print(f"    In span of R^3 basis? {in_full}")
    print(f"    In span of {{e1, e2}}?  {in_partial}")

    if c_full:
        recon = linear_combo(c_full, spanning_vecs)
        print(f"    Reconstruction error: {max(abs(a - b) for a, b in zip(w, recon)):.2e}")
    print()

print("Spanning set always contains every vector; partial set misses some.")

---

## 5. Exercises

Test your understanding with the following problems.

### Exercise 1 (Easy)

Are the vectors $\mathbf{v}_1 = [1, 2]$ and $\mathbf{v}_2 = [3, 6]$ linearly independent?

Predict the answer **before** running the code. What is the geometric relationship
between these vectors? Then verify using `is_independent`.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Given the set $S = \{[1,0,1,0],\; [0,1,0,1],\; [1,1,1,1],\; [2,1,2,1],\; [0,0,1,1]\}$
in $\mathbb{R}^4$:

1. Test if $S$ is linearly independent.
2. If dependent, find a **maximal linearly independent subset** by removing
   redundant vectors one at a time.
3. Verify your subset is independent and spans the same space as the original set.

*Hint:* A greedy approach works. Start with the first vector. Add the next vector
and test independence. If still independent, keep it. If dependent, skip it.
Continue until you have processed all vectors.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `remove_dependent(vectors)` that takes a list of vectors and
returns a **maximal linearly independent subset** that spans the same space as
the original set.

Your function should:
1. Process vectors in order.
2. Keep a vector if it is not in the span of the previously kept vectors.
3. Skip it if it is already in the span.
4. Return the independent subset.

Test on at least three different inputs:
- A set that is already independent.
- A set with exactly one redundant vector.
- A set where multiple vectors are redundant.

Verify that `is_independent(result)` returns `True` for each output.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Definition | Test |
|---|---|---|
| **Linear independence** | $c_1\mathbf{v}_1 + \cdots + c_k\mathbf{v}_k = \mathbf{0}$ only if all $c_i = 0$ | RREF of column matrix: every column is a pivot column |
| **Linear dependence** | $\exists\;$ nontrivial $c_1, \ldots, c_k$ with $c_1\mathbf{v}_1 + \cdots + c_k\mathbf{v}_k = \mathbf{0}$ | RREF has at least one free column |
| **Spanning set** | Every $\mathbf{w} \in V$ is a linear combination of the set | $\text{rank}(A) = \dim(V)$ where columns are the vectors |
| **Basis** | Independent spanning set | Both conditions above hold simultaneously |
| **Dimension bound** | $> n$ vectors in $\mathbb{R}^n$ must be dependent | Pigeonhole: more columns than rows means free variables |
| **Wronskian** | Determinant test for function independence | $W \neq 0$ at some point $\Rightarrow$ functions are independent |

**Takeaway:** Linear independence is about *non-redundancy*. A spanning set is about
*completeness*. A **basis** achieves both: every vector has a *unique* representation.
The connection to homogeneous systems ($A\mathbf{c} = \mathbf{0}$) makes these abstract
concepts computationally concrete through RREF.